# KPI 10 — Web Activity Quality

**requirements.md (Web & Traffic Analytics, KPI 10):** "Activity quality: 4xx/5xx error rates by endpoint."

This notebook creates the SQL view `b2b_ecommerce.gold.kpi_10_web_activity_quality` (idempotent: `CREATE OR REPLACE`). Views are catalog objects, not pipeline datasets — this notebook is **not** attached to `silver_to_gold_etl`; run the cells below once (or after any change to the fact/dim definitions).

**Definitions (state formulas):**

* **Grain:** one row per (**method**, **endpoint**) of **human** traffic — 386 rows. Method is kept in the grain because the same endpoint behaves differently per method (e.g. `POST /login` failed logins vs `GET /login` page views, `POST /checkout` step failures).
* **Bot exclusion (view policy):** bots excluded via `dim_device.is_bot` — the established KPI 8/9 view policy (the fact keeps bot traffic gross; 140,414 bot log rows are excluded here). NULL user-agent rows have no `dim_device` match and are retained as human. Rationale from pre-flight: 41,557 of the 57,269 gross 4xx rows (73%) are bot scanner traffic on attack paths (`/.env`, `/phpmyadmin/`, `/wp-login.php`, …) — not site activity quality — while 5xx is 89% human (2,705 of 3,029).
* **Measures:** `request_count`; `status_2xx_count`, `status_3xx_count`, `status_4xx_count`, `status_5xx_count` — sum exactly to `request_count` on every row (every fact status code is 2xx–5xx); `error_count` = 4xx + 5xx; `error_rate_pct` = ROUND(100 × error_count / request_count, 2); `error_4xx_rate_pct` / `error_5xx_rate_pct` on the same basis; `error_share_pct` = ROUND(100 × row error_count / all human errors, 2) — per-row ROUND, sums to ~100.00; `error_count_rank` = ROW_NUMBER by error_count DESC (tie-break method, endpoint — deterministic).
* **Verified:** 386 rows = 386 distinct pairs; 397,738 human requests (exact KPI 8/9 reconciliation: 538,152 gross − 140,414 bot rows); 18,417 human errors = 15,712 4xx + 2,705 5xx; 386/386 full matches via the independent recompute (anti-join bot exclusion + FLOOR(status_code/100) classification).

In [0]:
%sql
CREATE OR REPLACE VIEW b2b_ecommerce.gold.kpi_10_web_activity_quality
TBLPROPERTIES (
  'comment' = 'KPI 10 - Web activity quality (requirements.md: Activity quality: 4xx/5xx error rates by endpoint). Grain: one row per (method, endpoint) of HUMAN traffic (386 rows) - method kept in the grain because the same endpoint behaves differently per method (POST /login failed logins vs GET /login page views, POST /checkout step failures). Bots EXCLUDED via dim_device.is_bot (KPI 8/9 view policy; the fact keeps bot traffic gross - 140,414 bot log rows excluded, including 41,557 of the 57,269 gross 4xx which is bot scanner noise on attack paths like /.env, /phpmyadmin/, /wp-login.php); NULL-UA rows have no dim_device match and are retained as human. Measures: request_count; status_2xx_count/status_3xx_count/status_4xx_count/status_5xx_count sum exactly to request_count on every row (all fact status codes are 2xx-5xx); error_count = 4xx + 5xx; error_rate_pct = ROUND(100*error_count/request_count,2); error_4xx_rate_pct and error_5xx_rate_pct same basis; error_share_pct = ROUND(100*row error_count/all human errors,2) (per-row ROUND, sums ~100.00); error_count_rank = ROW_NUMBER by error_count DESC (tie-break method, endpoint - deterministic). Verified: 386 rows = 386 distinct (method, endpoint) pairs; 397,738 human requests = 538,152 gross - 140,414 bot rows (exact KPI 8/9 reconciliation); 18,417 human errors = 15,712 4xx + 2,705 5xx; 386/386 full matches on request_count, 4xx and 5xx counts via independent recompute (LEFT ANTI JOIN bot exclusion + FLOOR(status_code/100) classification).'
)
AS
WITH human_logs AS (
  SELECT l.method, l.endpoint, l.status_code
  FROM b2b_ecommerce.gold.fact_web_logs l
  LEFT JOIN b2b_ecommerce.gold.dim_device d
    ON l.user_agent = d.user_agent
  WHERE COALESCE(d.is_bot, false) = false
),
per_endpoint AS (
  SELECT
    method,
    endpoint,
    COUNT(*) AS request_count,
    SUM(CASE WHEN status_code BETWEEN 200 AND 299 THEN 1 ELSE 0 END) AS status_2xx_count,
    SUM(CASE WHEN status_code BETWEEN 300 AND 399 THEN 1 ELSE 0 END) AS status_3xx_count,
    SUM(CASE WHEN status_code BETWEEN 400 AND 499 THEN 1 ELSE 0 END) AS status_4xx_count,
    SUM(CASE WHEN status_code >= 500 THEN 1 ELSE 0 END) AS status_5xx_count
  FROM human_logs
  GROUP BY method, endpoint
)
SELECT
  method,
  endpoint,
  request_count,
  status_2xx_count,
  status_3xx_count,
  status_4xx_count,
  status_5xx_count,
  status_4xx_count + status_5xx_count AS error_count,
  ROUND(100 * (status_4xx_count + status_5xx_count) / request_count, 2) AS error_rate_pct,
  ROUND(100 * status_4xx_count / request_count, 2) AS error_4xx_rate_pct,
  ROUND(100 * status_5xx_count / request_count, 2) AS error_5xx_rate_pct,
  ROUND(100 * (status_4xx_count + status_5xx_count)
        / SUM(status_4xx_count + status_5xx_count) OVER (), 2) AS error_share_pct,
  ROW_NUMBER() OVER (
    ORDER BY (status_4xx_count + status_5xx_count) DESC, method ASC, endpoint ASC
  ) AS error_count_rank
FROM per_endpoint;

In [0]:
%sql
-- Verification: shape, totals, integrity. Expected: 386 rows = 386 distinct
-- (method, endpoint) pairs; SUM(request_count) = 397,738 (538,152 gross fact
-- rows - 140,414 bot rows, exact KPI 8/9 reconciliation);
-- SUM(status_4xx_count) = 15,712; SUM(status_5xx_count) = 2,705;
-- SUM(error_count) = 18,417; per-row class counts sum exactly to
-- request_count; error_count = 4xx + 5xx on every row; error_share_pct sums
-- ~100.00 +/- 0.06 (per-row ROUND); error_count_rank contiguous 1..386 and
-- identical to a fresh recompute; rates recompute exactly; 0 null keys.
WITH ranked AS (
  SELECT v.*,
         ROW_NUMBER() OVER (
           ORDER BY error_count DESC, method ASC, endpoint ASC
         ) AS rank_recompute
  FROM b2b_ecommerce.gold.kpi_10_web_activity_quality v
)
SELECT
  COUNT(*) AS rows_method_endpoint,
  COUNT(DISTINCT method || '|' || endpoint) AS distinct_pairs,
  SUM(request_count) AS total_requests,
  SUM(status_2xx_count) AS total_2xx,
  SUM(status_3xx_count) AS total_3xx,
  SUM(status_4xx_count) AS total_4xx,
  SUM(status_5xx_count) AS total_5xx,
  SUM(error_count) AS total_errors,
  SUM(CASE WHEN status_2xx_count + status_3xx_count + status_4xx_count + status_5xx_count <> request_count
           THEN 1 ELSE 0 END) AS decomposition_mismatches,
  SUM(CASE WHEN error_count <> status_4xx_count + status_5xx_count
           THEN 1 ELSE 0 END) AS error_mismatches,
  ROUND(SUM(error_share_pct), 2) AS share_sum,
  COUNT(DISTINCT error_count_rank) AS distinct_ranks,
  MAX(error_count_rank) AS max_rank,
  SUM(CASE WHEN error_count_rank <> rank_recompute
           THEN 1 ELSE 0 END) AS rank_mismatches,
  SUM(CASE WHEN method IS NULL OR endpoint IS NULL
           THEN 1 ELSE 0 END) AS null_keys,
  SUM(CASE WHEN error_rate_pct <> ROUND(100 * error_count / request_count, 2)
            OR error_4xx_rate_pct <> ROUND(100 * status_4xx_count / request_count, 2)
            OR error_5xx_rate_pct <> ROUND(100 * status_5xx_count / request_count, 2)
           THEN 1 ELSE 0 END) AS rate_mismatches
FROM ranked;

In [0]:
%sql
-- Verification: independently recompute every (method, endpoint) cell through
-- a different path - bot rows excluded via LEFT ANTI JOIN on the is_bot dim
-- subset (vs LEFT JOIN + COALESCE filter), status classes derived via
-- FLOOR(status_code / 100) (vs BETWEEN ranges) - then FULL OUTER JOIN against
-- the stored view and compare the three count measures (rates follow
-- arithmetically from them). Expected: 386 rows compared, 386/386 full
-- matches, 0 rows only on either side.
WITH bots AS (
  SELECT user_agent
  FROM b2b_ecommerce.gold.dim_device
  WHERE is_bot = true
),
human2 AS (
  SELECT l.method, l.endpoint, FLOOR(l.status_code / 100) AS status_class
  FROM b2b_ecommerce.gold.fact_web_logs l
  LEFT ANTI JOIN bots b
    ON l.user_agent = b.user_agent
),
recomputed AS (
  SELECT
    method,
    endpoint,
    COUNT(*) AS request_count,
    SUM(CASE WHEN status_class = 4 THEN 1 ELSE 0 END) AS error_4xx_count,
    SUM(CASE WHEN status_class = 5 THEN 1 ELSE 0 END) AS error_5xx_count
  FROM human2
  GROUP BY method, endpoint
),
compare AS (
  SELECT v.method AS v_method, v.endpoint AS v_endpoint,
         v.request_count AS v_requests, v.status_4xx_count AS v_4xx, v.status_5xx_count AS v_5xx,
         r.method AS r_method, r.endpoint AS r_endpoint,
         r.request_count AS r_requests, r.error_4xx_count AS r_4xx, r.error_5xx_count AS r_5xx
  FROM b2b_ecommerce.gold.kpi_10_web_activity_quality v
  FULL OUTER JOIN recomputed r
    ON v.method = r.method AND v.endpoint = r.endpoint
)
SELECT
  COUNT(*) AS rows_compared,
  SUM(CASE WHEN v_method IS NOT NULL AND r_method IS NOT NULL
            AND v_requests = r_requests
            AND v_4xx = r_4xx
            AND v_5xx = r_5xx
           THEN 1 ELSE 0 END) AS full_matches,
  SUM(CASE WHEN v_method IS NULL OR r_method IS NULL
           THEN 1 ELSE 0 END) AS one_sided_rows
FROM compare;

In [0]:
%sql
-- Headline: most problematic endpoints first (top 25 by error_count_rank).
SELECT error_count_rank, method, endpoint, request_count,
       error_count, error_rate_pct,
       error_4xx_rate_pct, error_5xx_rate_pct, error_share_pct
FROM b2b_ecommerce.gold.kpi_10_web_activity_quality
ORDER BY error_count_rank
LIMIT 25